# Phase 0: the LangGraph lab

Does LangGraph give the OpenSec Analyst tool calling, structured answers, follow-up memory and
MLflow tracing, on AMD's and NVIDIA's real filings? The question, method and pass bar are in
`labs/BRIEF.md`. Run it top to bottom; reruns use the cache in `data/filings/`.

In [1]:
import hashlib
import json
import re
import statistics
import time
import uuid
import warnings
from collections import defaultdict, deque
from collections.abc import Callable
from dataclasses import dataclass
from datetime import date
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from typing import Any, Literal

import httpx
import mlflow
import numpy as np
import pandas as pd
import yaml
from bs4 import BeautifulSoup, Tag, XMLParsedAsHTMLWarning
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
from openai import OpenAI
from pydantic import BaseModel, Field
from pydantic_settings import BaseSettings, SettingsConfigDict
from rank_bm25 import BM25Okapi

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
FILINGS_DIR = DATA_DIR / "filings"


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=REPO_ROOT / ".env", env_ignore_empty=True, extra="ignore"
    )

    sec_user_agent: str
    openai_api_key: str
    model_name: str = "gpt-6.1-terra"
    model_reasoning_effort: str = "medium"
    embedding_model: str = "text-embedding-3-large"
    mlflow_tracking_uri: str = "http://localhost:5001"
    price_input_per_mtok: Decimal | None = None
    price_output_per_mtok: Decimal | None = None


settings = Settings()  # pyright: ignore[reportCallIssue]
companies = yaml.safe_load((DATA_DIR / "tickers.yaml").read_text())["companies"]
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("phase-0-langgraph-lab")
mlflow.langchain.autolog()
pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)  # some filings are XHTML

## 1. Fetch filings from EDGAR

Every request waits for a slot in a sliding one-second window of at most 9 requests, and sends
the User-Agent from `.env`. The window is the last 3 completed fiscal years' 10-Ks and 10-Qs,
plus this fiscal year's 10-Qs (plan decision 4).

In [2]:
MAX_REQUESTS_PER_SECOND = 9


class RateLimiter:
    """Blocks until a request fits in a sliding one-second window."""

    def __init__(self, max_per_second: int = MAX_REQUESTS_PER_SECOND) -> None:
        self._max = max_per_second
        self._sent: deque[float] = deque()

    def wait(self) -> float:
        while True:
            now = time.monotonic()
            while self._sent and now - self._sent[0] >= 1.0:
                self._sent.popleft()
            if len(self._sent) < self._max:
                self._sent.append(now)
                return now
            time.sleep(1.0 - (now - self._sent[0]))


def max_requests_in_any_second(sent_at: list[float]) -> int:
    times = sorted(sent_at)
    most, start = 0, 0
    for end, moment in enumerate(times):
        while moment - times[start] >= 1.0:
            start += 1
        most = max(most, end - start + 1)
    return most


class EdgarClient:
    def __init__(self, user_agent: str) -> None:
        self._limiter = RateLimiter()
        self._http = httpx.Client(
            headers={"User-Agent": user_agent, "Accept-Encoding": "gzip, deflate"}, timeout=60
        )
        self.sent_at: list[float] = []

    def get(self, url: str) -> str:
        self.sent_at.append(self._limiter.wait())
        response = self._http.get(url)
        response.raise_for_status()
        return response.text


def cached(path: Path, fetch: Callable[[], str]) -> str:
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(fetch())
    return path.read_text()


@dataclass(frozen=True)
class Filing:
    ticker: str
    cik: int
    form: str
    accession: str
    filing_date: date
    report_date: date
    primary_document: str


def filings_from_submissions(ticker: str, cik: int, submissions: dict[str, Any]) -> list[Filing]:
    recent = submissions["filings"]["recent"]
    rows = zip(
        recent["form"],
        recent["accessionNumber"],
        recent["filingDate"],
        recent["reportDate"],
        recent["primaryDocument"],
        strict=True,
    )
    return [
        Filing(ticker, cik, form, accn, date.fromisoformat(filed), date.fromisoformat(end), doc)
        for form, accn, filed, end, doc in rows
        if form in ("10-K", "10-Q")
    ]


def filings_in_window(filings: list[Filing], fiscal_years: int = 3) -> list[Filing]:
    annual = sorted((f.report_date for f in filings if f.form == "10-K"), reverse=True)
    start = annual[fiscal_years] if len(annual) > fiscal_years else date.min
    return sorted((f for f in filings if f.report_date > start), key=lambda f: f.report_date)


edgar = EdgarClient(settings.sec_user_agent)
filings: list[Filing] = []
companyfacts: dict[str, dict[str, Any]] = {}
for company in companies:
    ticker, cik = company["ticker"], company["cik"]
    folder = FILINGS_DIR / ticker
    submissions = json.loads(
        cached(
            folder / "submissions.json",
            lambda cik=cik: edgar.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json"),
        )
    )
    companyfacts[ticker] = json.loads(
        cached(
            folder / "companyfacts.json",
            lambda cik=cik: edgar.get(
                f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"
            ),
        )
    )
    for filing in filings_in_window(filings_from_submissions(ticker, cik, submissions)):
        url = (
            f"https://www.sec.gov/Archives/edgar/data/{cik}/"
            f"{filing.accession.replace('-', '')}/{filing.primary_document}"
        )
        cached(folder / f"{filing.accession}.htm", lambda url=url: edgar.get(url))
        filings.append(filing)

request_log = FILINGS_DIR / "request_log.json"
if edgar.sent_at:
    request_log.write_text(json.dumps(edgar.sent_at))
logged = json.loads(request_log.read_text()) if request_log.exists() else []
print(f"Requests this run: {len(edgar.sent_at)}; logged: {len(logged)}")
print(f"Most requests in any one second: {max_requests_in_any_second(logged)}")
pd.DataFrame(
    [(f.ticker, f.form, f.report_date, f.filing_date, f.accession) for f in filings],
    columns=["ticker", "form", "period end", "filed", "accession"],
)

Requests this run: 0; logged: 33
Most requests in any one second: 9


,ticker,form,period end,filed,accession
0,AMD,10-Q,2023-04-01,2023-05-03,0000002488-23-000076
1,AMD,10-Q,2023-07-01,2023-08-02,0000002488-23-000139
2,AMD,10-Q,2023-09-30,2023-11-01,0000002488-23-000195
3,AMD,10-K,2023-12-30,2024-01-31,0000002488-24-000012
4,AMD,10-Q,2024-03-30,2024-05-01,0000002488-24-000056
5,AMD,10-Q,2024-06-29,2024-07-31,0000002488-24-000123
6,AMD,10-Q,2024-09-28,2024-10-30,0000002488-24-000163
7,AMD,10-K,2024-12-28,2025-02-05,0000002488-25-000012
8,AMD,10-Q,2025-03-29,2025-05-07,0000002488-25-000047
9,AMD,10-Q,2025-06-28,2025-08-06,0000002488-25-000108


## 2. XBRL facts, fiscal labels and restatements

Companyfacts tags each value with the fiscal year of the filing that reported it, not of the
period itself. So each period's label comes from the filing that closes it: the latest period end
in a filing is that filing's own fiscal period. A value is flagged as revised when a later filing
reports a different number for the same concept and period (plan decision 8); a value whose sign
alone flipped is a tagging change and isn't flagged.

In [3]:
@dataclass(frozen=True)
class FiscalPeriod:
    year: int
    period: str  # FY, Q1, Q2, Q3


@dataclass(frozen=True)
class Fact:
    id: str
    ticker: str
    concept: str
    label: str
    unit: str
    start: date | None
    end: date
    value: Decimal
    accession: str
    form: str
    period_label: str
    restated: bool
    earlier_value: Decimal | None


def _entries(facts: dict[str, Any], taxonomies: tuple[str, ...]) -> list[dict[str, Any]]:
    return [
        entry
        for taxonomy in taxonomies
        for body in facts["facts"].get(taxonomy, {}).values()
        for entries in body["units"].values()
        for entry in entries
        if entry["form"] in ("10-K", "10-Q") and entry.get("fp")
    ]


def fiscal_periods(facts: dict[str, Any]) -> dict[date, FiscalPeriod]:
    """Maps each period end to the fiscal period of the filing that closes it."""
    latest: dict[str, tuple[date, FiscalPeriod]] = {}
    # dei cover-page values are dated after the period ends, so only us-gaap sets the map.
    for entry in _entries(facts, ("us-gaap",)):
        end = date.fromisoformat(entry["end"])
        if entry["accn"] not in latest or end > latest[entry["accn"]][0]:
            latest[entry["accn"]] = (end, FiscalPeriod(entry["fy"], entry["fp"]))
    return dict(latest.values())


def period_label(start: date | None, end: date, periods: dict[date, FiscalPeriod]) -> str:
    """FY2025, Q3 FY2026, 9M FY2026, or "end of FY2025" for balance-sheet values."""
    fiscal = periods.get(end)
    if fiscal is None:
        return f"{start} to {end}" if start else f"as of {end}"
    closing = f"FY{fiscal.year}" if fiscal.period == "FY" else f"{fiscal.period} FY{fiscal.year}"
    if start is None:
        return f"end of {closing}"
    days = (end - start).days
    if 350 <= days <= 380:
        return f"FY{fiscal.year}"
    if 80 <= days <= 100:
        quarter = "Q4" if fiscal.period == "FY" else fiscal.period
        return f"{quarter} FY{fiscal.year}"
    for middle, months in ((180, 6), (270, 9)):
        if abs(days - middle) <= 12:
            return f"{months}M FY{fiscal.year}"
    return f"{start} to {end}"


def load_facts(ticker: str, facts: dict[str, Any]) -> list[Fact]:
    periods = fiscal_periods(facts)
    loaded = []
    for taxonomy in ("us-gaap", "dei"):
        for concept, body in facts["facts"].get(taxonomy, {}).items():
            for unit, entries in body["units"].items():
                groups: dict[tuple[str | None, str], list[dict[str, Any]]] = defaultdict(list)
                for entry in entries:
                    if entry["form"] in ("10-K", "10-Q"):
                        groups[(entry.get("start"), entry["end"])].append(entry)
                for (start, end), group in groups.items():
                    by_filing = sorted(group, key=lambda e: (e["filed"], e["accn"]))
                    values = [Decimal(str(e["val"])) for e in by_filing]
                    latest = values[-1]
                    # A flipped sign is a tagging change, not a revision.
                    earlier = [v for v in values if abs(v) != abs(latest)]
                    source = by_filing[values.index(latest)]
                    start_day = date.fromisoformat(start) if start else None
                    end_day = date.fromisoformat(end)
                    key = f"{ticker}:{concept}:{unit}:{start}:{end}"
                    loaded.append(
                        Fact(
                            id="f_" + hashlib.sha1(key.encode()).hexdigest()[:8],
                            ticker=ticker,
                            concept=concept,
                            label=body["label"] or concept,
                            unit=unit,
                            start=start_day,
                            end=end_day,
                            value=latest,
                            accession=source["accn"],
                            form=source["form"],
                            period_label=period_label(start_day, end_day, periods),
                            restated=bool(earlier),
                            earlier_value=earlier[-1] if earlier else None,
                        )
                    )
    return loaded


def _words(text: str) -> set[str]:
    spaced = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    return {w.rstrip("s") for w in re.findall(r"[a-z]+", spaced.lower()) if len(w) > 2}


class FactStore:
    def __init__(self, facts: list[Fact]) -> None:
        self._by_id = {f.id: f for f in facts}
        self._concept_size: dict[tuple[str, str], int] = defaultdict(int)
        for f in facts:
            self._concept_size[(f.ticker, f.concept)] += 1

    def get(self, fact_id: str) -> Fact | None:
        return self._by_id.get(fact_id)

    def lookup(self, ticker: str, concept: str, period: str) -> Fact:
        return next(
            f
            for f in self._by_id.values()
            if (f.ticker, f.concept, f.period_label) == (ticker, concept, period)
            and f.unit in ("USD", "USD/shares", "shares", "pure")
        )

    def find(self, ticker: str, query: str, period: str | None, limit: int = 12) -> list[Fact]:
        """Facts whose concept best matches the query, most-reported concepts first."""
        wanted = _words(query)

        def score(f: Fact) -> tuple[int, int, int]:
            words = _words(f.label) | _words(f.concept)
            overlap = len(wanted & words)
            return (
                overlap,
                -len(_words(f.label) - wanted),
                self._concept_size[(f.ticker, f.concept)],
            )

        candidates = [
            f
            for f in self._by_id.values()
            if f.ticker == ticker.upper()
            and (period is None or f.period_label in (period, f"end of {period}"))
            and score(f)[0]
        ]
        return sorted(candidates, key=score, reverse=True)[:limit]


facts = FactStore([f for t, body in companyfacts.items() for f in load_facts(t, body)])
periods_by_ticker = {t: fiscal_periods(body) for t, body in companyfacts.items()}


def filing_label(filing: Filing) -> str:
    """ "2025" for a 10-K, "Q3 FY2026" for a 10-Q, in the company's own fiscal calendar."""
    fiscal = periods_by_ticker[filing.ticker][filing.report_date]
    return str(fiscal.year) if filing.form == "10-K" else f"{fiscal.period} FY{fiscal.year}"


filing_chips = {f.accession: f"{f.ticker} {f.form} {filing_label(f)}" for f in filings}
pd.DataFrame(
    [(f.ticker, f.form, f.report_date, filing_chips[f.accession]) for f in filings],
    columns=["ticker", "form", "period end", "chip"],
)

,ticker,form,period end,chip
0,AMD,10-Q,2023-04-01,AMD 10-Q Q1 FY2023
1,AMD,10-Q,2023-07-01,AMD 10-Q Q2 FY2023
2,AMD,10-Q,2023-09-30,AMD 10-Q Q3 FY2023
3,AMD,10-K,2023-12-30,AMD 10-K 2023
4,AMD,10-Q,2024-03-30,AMD 10-Q Q1 FY2024
5,AMD,10-Q,2024-06-29,AMD 10-Q Q2 FY2024
6,AMD,10-Q,2024-09-28,AMD 10-Q Q3 FY2024
7,AMD,10-K,2024-12-28,AMD 10-K 2024
8,AMD,10-Q,2025-03-29,AMD 10-Q Q1 FY2025
9,AMD,10-Q,2025-06-28,AMD 10-Q Q2 FY2025


## 3. Passages: text by Item, then chunks

Each filing becomes plain text with one block per paragraph or table. Item headings split it into
sections, and each section into chunks of about 2,000 characters. A chunk keeps its ticker, form,
fiscal period and Item, for chips like `AMD 10-K 2025 · Item 7` (plan decision 6).

In [4]:
BLOCK_TAGS = ["p", "div", "li", "h1", "h2", "h3", "h4", "h5", "h6"]
OPENING_CELLS = {"$", "(", "($"}
CLOSING_CELLS = {"%", ")", "%)"}
PART_HEADING = re.compile(r"^PART\s+(IV|III|II|I)\b", re.IGNORECASE)
ITEM_HEADING = re.compile(r"^ITEM\s+(\d{1,2}[A-C]?)\s*[.:\-\u2013\u2014]?", re.IGNORECASE)
MAX_HEADING_LENGTH = 120
MIN_SECTION_LENGTH = 300
CHUNK_SIZE = 2000


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def _row_text(cells: list[str]) -> str:
    """Joins a table row, keeping "$", "(" and "%" attached to their numbers."""
    joined: list[str] = []
    for cell in cells:
        if joined and (cell in CLOSING_CELLS or joined[-1] in OPENING_CELLS):
            joined[-1] += cell
        else:
            joined.append(cell)
    return " | ".join(joined)


def _table_text(table: Tag) -> str:
    rows = []
    for row in table.find_all("tr"):
        cells = [_clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]
        if any(cells):
            rows.append(_row_text([c for c in cells if c]))
    return "\n".join(rows)


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for hidden in soup.select('[style*="display:none"], [style*="display: none"]'):
        hidden.decompose()
    for table in soup.find_all("table"):
        replacement = soup.new_tag("p", attrs={"data-table": "1"})
        replacement.string = _table_text(table)
        table.replace_with(replacement)
    blocks = []
    for element in soup.find_all(BLOCK_TAGS):
        if element.find(BLOCK_TAGS):
            continue
        is_table = element.has_attr("data-table")
        text = element.get_text() if is_table else _clean(element.get_text(" "))
        if text:
            blocks.append(text)
    return "\n\n".join(blocks)


def split_sections(text: str, form: str) -> list[tuple[str, int, int]]:
    """(Item, start, end) from each Item heading to the next. 10-Q Items carry their Part."""
    headings: list[tuple[int, str]] = [(0, "Cover")]
    part, position = "", 0
    for block in text.split("\n\n"):
        if len(block) <= MAX_HEADING_LENGTH and "\n" not in block:
            if part_match := PART_HEADING.match(block):
                part = part_match.group(1).upper()
            elif item_match := ITEM_HEADING.match(block):
                item = f"Item {item_match.group(1).upper()}"
                headings.append((position, f"Part {part}, {item}" if form == "10-Q" else item))
        position += len(block) + 2
    ends = [start for start, _ in headings[1:]] + [len(text)]
    sections = [(item, start, end) for (start, item), end in zip(headings, ends, strict=True)]
    return [s for s in sections if s[2] - s[1] >= MIN_SECTION_LENGTH]


@dataclass(frozen=True)
class Passage:
    id: str
    ticker: str
    form: str
    fiscal_year: int
    item: str
    accession: str
    text: str

    @property
    def chip(self) -> str:
        return f"{filing_chips[self.accession]} · {self.item}"


def passages_for(filing: Filing, text: str) -> list[Passage]:
    fiscal_year = periods_by_ticker[filing.ticker][filing.report_date].year
    result = []
    for item, section_start, section_end in split_sections(text, filing.form):
        start = end = section_start
        for block in re.finditer(r"[^\n]+(?:\n(?!\n)[^\n]+)*", text[section_start:section_end]):
            block_end = section_start + block.end()
            if end > start and block_end - start > CHUNK_SIZE:
                result.append((item, start, end))
                start = section_start + block.start()
            end = block_end
        if end > start:
            result.append((item, start, end))
    return [
        Passage(
            "p_" + hashlib.sha1(f"{filing.accession}:{start}".encode()).hexdigest()[:10],
            filing.ticker,
            filing.form,
            fiscal_year,
            item,
            filing.accession,
            text[start:end],
        )
        for item, start, end in result
    ]


passages: list[Passage] = []
for filing in filings:
    folder = FILINGS_DIR / filing.ticker
    text = cached(
        folder / f"{filing.accession}.txt",
        lambda f=folder, a=filing.accession: html_to_text((f / f"{a}.htm").read_text()),
    )
    passages.extend(passages_for(filing, text))

summary = pd.DataFrame(
    [(p.ticker, p.form, p.item) for p in passages], columns=["ticker", "form", "item"]
)
print(f"{len(passages)} passages")
summary.value_counts().unstack("ticker").fillna(0).astype(int).head(40)

3888 passages


ticker                 AMD  NVDA
form item                       
10-Q Cover              24    33
     Part I, Item 1    318   336
     Part I, Item 2    199   220
     Part I, Item 3     12    11
     Part I, Item 4     15    18
     Part II, Item 1A  933   287
     Part II, Item 2    13    22
     Part II, Item 6    23    22
     Part II, Item 5    12     9
     Part II, Item 1     0     9
10-K Cover              12    21
     Item 1            115    94
     Item 1A           239   209
     Item 1C             9     8
     Item 2              3     3
     Item 5              6    10
     Item 7             63    66
     Item 7A            10     7
     Item 8            194     0
     Item 9A            12     6
     Item 9B             3     3
     Item 10             3     6
     Item 11             3     3
     Item 12             3     3
     Item 14             3     3
     Item 15            40   197
     Item 16             3     6
     Item 9C             0     3
     Item 13             0     3

## 4. Hybrid search

OpenAI embeddings (cached on disk) plus BM25 keywords, merged by reciprocal rank fusion
(plan decision 7). Phase 1 moves this to pgvector and Postgres full-text.

In [5]:
RRF_K = 60
EMBED_BATCH = 128
EMBED_MAX_CHARS = 16_000


class OpenAIEmbedder:
    def __init__(self, api_key: str, model: str, cache_dir: Path) -> None:
        self._client = OpenAI(api_key=api_key)
        self._model = model
        self._cache_dir = cache_dir / model
        self._cache_dir.mkdir(parents=True, exist_ok=True)

    def embed(self, texts: list[str], cache: bool) -> np.ndarray:
        if not cache:
            return self._call(texts)
        keys = [hashlib.sha1(t.encode()).hexdigest() for t in texts]
        missing = [i for i, k in enumerate(keys) if not (self._cache_dir / f"{k}.npy").exists()]
        for begin in range(0, len(missing), EMBED_BATCH):
            batch = missing[begin : begin + EMBED_BATCH]
            for i, vector in zip(batch, self._call([texts[i] for i in batch]), strict=True):
                np.save(self._cache_dir / f"{keys[i]}.npy", vector)
        return np.stack([np.load(self._cache_dir / f"{k}.npy") for k in keys])

    def _call(self, texts: list[str]) -> np.ndarray:
        # The model reads up to 8,191 tokens; a few very long tables are cut for embedding only.
        inputs = [text[:EMBED_MAX_CHARS] for text in texts]
        result = self._client.embeddings.create(model=self._model, input=inputs)
        return np.array([item.embedding for item in result.data], dtype=np.float32)


def _tokens(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


class PassageIndex:
    def __init__(self, passages: list[Passage], embedder: OpenAIEmbedder) -> None:
        self.passages = passages
        self._by_id = {p.id: p for p in passages}
        self._embedder = embedder
        vectors = embedder.embed([p.text for p in passages], cache=True)
        self._vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
        self._keywords = BM25Okapi([_tokens(p.text) for p in passages])

    def get(self, passage_id: str) -> Passage | None:
        return self._by_id.get(passage_id)

    def search(
        self,
        query: str,
        ticker: str | None,
        form: str | None,
        fiscal_year: int | None,
        limit: int = 6,
    ) -> list[Passage]:
        allowed = np.array(
            [
                (ticker is None or p.ticker == ticker.upper())
                and (form is None or p.form == form.upper())
                and (fiscal_year is None or p.fiscal_year == fiscal_year)
                for p in self.passages
            ]
        )
        if not allowed.any():
            return []
        query_vector = self._embedder.embed([query], cache=False)[0]
        similarity = self._vectors @ (query_vector / np.linalg.norm(query_vector))
        keyword = np.asarray(self._keywords.get_scores(_tokens(query)))
        fused = np.zeros(len(self.passages))
        for scores in (similarity, keyword):
            order = np.argsort(-np.where(allowed, scores, -np.inf))
            ranks = np.empty(len(order))
            ranks[order] = np.arange(1, len(order) + 1)
            fused += 1.0 / (RRF_K + ranks)
        fused[~allowed] = -1.0
        return [self.passages[i] for i in np.argsort(-fused)[: min(limit, int(allowed.sum()))]]


started = time.perf_counter()
embedder = OpenAIEmbedder(
    settings.openai_api_key, settings.embedding_model, FILINGS_DIR / "embeddings"
)
index = PassageIndex(passages, embedder)
print(f"Indexed {len(passages)} passages in {time.perf_counter() - started:.1f} s")
[p.chip for p in index.search("export controls on data center GPUs to China", "NVDA", "10-K", None)]

Indexed 3888 passages in 1.1 s


['NVDA 10-K 2026 · Item 1A',
 'NVDA 10-K 2026 · Item 1',
 'NVDA 10-K 2026 · Item 1A',
 'NVDA 10-K 2025 · Item 7',
 'NVDA 10-K 2025 · Item 1A',
 'NVDA 10-K 2026 · Item 1A']

## 5. Answers: figures, math and the citation check

The model writes figures only as `{fact:ID}` or `{calc:ID}`; Python formats them (R-05, R-07,
R-08). Math runs in the `compute` tool. Before an answer is shown, it must have a table or bullets
after the summary (R-02), every passage must exist and every quote must appear in its passage (R-03). A figure typed as text must appear in a cited quote.

In [6]:
MILLION = Decimal(1_000_000)
NOT_DISCLOSED = "Not disclosed in the filings."
PLACEHOLDER = re.compile(r"\{(fact|calc):([a-z0-9_]+)\}")
FIGURE = re.compile(
    r"\$\s?\d[\d,]*(?:\.\d+)?|\b\d[\d,]*(?:\.\d+)?\s?%|\b\d{1,3}(?:,\d{3})+(?:\.\d+)?"
    r"|\b\d+\.\d+|\b\d+(?:\.\d+)?\s?(?:million|billion|thousand)\b"
)
Operation = Literal["change", "percent_change", "ratio", "sum"]


def _rounded(value: Decimal, places: str) -> Decimal:
    return value.quantize(Decimal(places), rounding=ROUND_HALF_UP)


def _magnitude(value: Decimal, unit: str) -> str:
    size = abs(value)
    if unit == "USD":
        return (
            f"${_rounded(size / MILLION, '0.1'):,}M"
            if size >= MILLION
            else f"${_rounded(size, '1'):,}"
        )
    if unit == "USD/shares":
        return f"${_rounded(size, '0.01'):,}"
    if unit == "percent":
        return f"{_rounded(size, '0.1'):,}%"
    if unit == "pure":
        return f"{_rounded(size * 100, '0.1'):,}%"
    if unit == "shares" and size >= MILLION:
        return f"{_rounded(size / MILLION, '0.1'):,}M shares"
    return f"{_rounded(size, '1'):,} {unit}"


def format_value(value: Decimal, unit: str) -> str:
    return f"({_magnitude(value, unit)})" if value < 0 else _magnitude(value, unit)


def format_change(value: Decimal, unit: str) -> str:
    return format_value(value, unit) if value < 0 else f"+{_magnitude(value, unit)}"


@dataclass(frozen=True)
class Calculation:
    id: str
    operation: Operation
    facts: tuple[Fact, ...]
    value: Decimal
    unit: str
    period_label: str


def calculate(operation: Operation, inputs: list[Fact]) -> Calculation:
    """change and percent_change go from the first fact to the second; ratio is first / second."""
    units = {f.unit for f in inputs}
    if operation != "ratio" and len(units) > 1:
        raise ValueError(f"facts have different units: {sorted(units)}")
    if operation == "sum":
        value, unit, label = (
            sum((f.value for f in inputs), Decimal(0)),
            inputs[0].unit,
            inputs[0].period_label,
        )
    elif len(inputs) != 2:
        raise ValueError(f"{operation} takes exactly two facts")
    else:
        first, second = inputs
        if first.ticker != second.ticker:
            label = f"{first.ticker} {first.period_label} vs {second.ticker} {second.period_label}"
        elif first.period_label != second.period_label:
            label = f"{first.period_label} to {second.period_label}"
        else:
            label = first.period_label
        if operation == "change":
            value, unit = second.value - first.value, first.unit
        elif operation == "percent_change":
            value, unit = (second.value - first.value) / abs(first.value) * 100, "percent"
        else:
            value, unit = first.value / second.value * 100, "percent"
    key = f"{operation}:{[f.id for f in inputs]}"
    return Calculation(
        "c_" + hashlib.sha1(key.encode()).hexdigest()[:8],
        operation,
        tuple(inputs),
        value,
        unit,
        label,
    )


class Citation(BaseModel):
    passage_id: str = Field(description="A passage id from search_passages, such as p_1a2b3c4d5e")
    quote: str = Field(description="A short quote copied exactly from that passage")


class Claim(BaseModel):
    text: str = Field(description="One sentence. Write every figure as {fact:ID} or {calc:ID}.")
    citations: list[Citation] = Field(description="Passages that support the claim")


class Table(BaseModel):
    columns: list[str]
    rows: list[list[str]] = Field(description="Cells hold labels or {fact:ID} / {calc:ID}")


class Answer(BaseModel):
    """The OpenSec Analyst's answer: a summary, then a table or bullets, then caveats."""

    kind: Literal["answer", "not_disclosed", "refusal"]
    summary: Claim
    table: Table | None
    bullets: list[Claim]
    caveats: list[str]


calculations: dict[str, Calculation] = {}


def _squash(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().lower()


def _references(texts: list[str]) -> tuple[list[Fact], list[Calculation], list[str]]:
    found_facts, found_calcs, unknown = [], [], []
    for text in texts:
        for kind, ref in PLACEHOLDER.findall(text):
            item = facts.get(ref) if kind == "fact" else calculations.get(ref)
            if item is None:
                unknown.append(f"{kind}:{ref}")
            elif isinstance(item, Fact):
                found_facts.append(item)
            else:
                found_calcs.append(item)
    return found_facts, found_calcs, unknown


def _typed_figures(text: str, quotes: list[str]) -> list[str]:
    quoted = " ".join(_squash(q) for q in quotes)
    return [m for m in FIGURE.findall(PLACEHOLDER.sub("", text)) if _squash(m) not in quoted]


def answer_texts(answer: Answer) -> list[str]:
    cells = [cell for row in answer.table.rows for cell in row] if answer.table else []
    return [answer.summary.text, *(b.text for b in answer.bullets), *cells, *answer.caveats]


def check_answer(answer: Answer) -> list[str]:
    """Problems that stop the answer being shown. Empty means it passes."""
    if answer.kind != "answer":
        return []
    problems = []
    if not answer.table and not answer.bullets:
        problems.append("After the summary, add a table or at least one bullet (R-02).")
    claims = [answer.summary, *answer.bullets]
    all_quotes = [c.quote for claim in claims for c in claim.citations]
    for claim in claims:
        if not claim.citations and not PLACEHOLDER.search(claim.text):
            problems.append(f"No citation for: {claim.text}")
        for citation in claim.citations:
            passage = index.get(citation.passage_id)
            if passage is None:
                problems.append(f"Unknown passage {citation.passage_id}")
            elif _squash(citation.quote) not in _squash(passage.text):
                problems.append(f"Quote not in {citation.passage_id}: {citation.quote}")
        quotes = [c.quote for c in claim.citations]
        problems += [
            f"Typed figure {f} is not from a fact, calculation or quote"
            for f in _typed_figures(claim.text, quotes)
        ]
    others = answer_texts(answer)[1 + len(answer.bullets) :]
    for text in others:
        problems += [
            f"Typed figure {f} is not from a fact, calculation or quote"
            for f in _typed_figures(text, all_quotes)
        ]
    _, _, unknown = _references(answer_texts(answer))
    problems += [f"Unknown id {ref}" for ref in unknown]
    return problems


def _fact_chip(fact: Fact) -> str:
    return (
        f"{filing_chips.get(fact.accession, f'{fact.ticker} {fact.form} {fact.accession}')} · XBRL"
    )


def _render_text(text: str, row: list[str] | None) -> str:
    """Fills in figures. In a table row, the period is left out when the row already shows it."""

    def figure(match: re.Match[str]) -> str:
        kind, ref = match.groups()
        if kind == "fact":
            fact = facts.get(ref)
            assert fact is not None
            value, label = format_value(fact.value, fact.unit), fact.period_label
        else:
            calc = calculations[ref]
            changed = calc.operation in ("change", "percent_change")
            value = (
                format_change(calc.value, calc.unit)
                if changed
                else format_value(calc.value, calc.unit)
            )
            label = calc.period_label
        if row is not None:
            return value if any(label in cell for cell in row) else f"{value} ({label})"
        if label.startswith("end of"):
            return f"{value} at the {label}"
        if " vs " in label:
            return f"{value} ({label})"
        return f"{value} from {label}" if " to " in label else f"{value} in {label}"

    return PLACEHOLDER.sub(figure, text)


def _chips(claim: Claim) -> str:
    cited_facts, cited_calcs, _ = _references([claim.text])
    for calc in cited_calcs:
        cited_facts.extend(calc.facts)
    chips = [index.get(c.passage_id).chip for c in claim.citations if index.get(c.passage_id)]  # pyright: ignore[reportOptionalMemberAccess]
    chips += [_fact_chip(f) for f in cited_facts]
    return " ".join(f"`{chip}`" for chip in dict.fromkeys(chips))


def render_answer(answer: Answer) -> str:
    """Markdown: summary, then a table or bullets, then caveats, each claim ending in chips."""
    if answer.kind == "not_disclosed":
        return NOT_DISCLOSED
    if answer.kind == "refusal":
        return answer.summary.text
    lines = [f"{_render_text(answer.summary.text, None)} {_chips(answer.summary)}", ""]
    if answer.table:
        lines += [
            "| " + " | ".join(answer.table.columns) + " |",
            "|" + " --- |" * len(answer.table.columns),
        ]
        lines += [
            "| " + " | ".join(_render_text(c, row) for c in row) + " |" for row in answer.table.rows
        ]
        lines.append("")
    lines += [f"- {_render_text(b.text, None)} {_chips(b)}" for b in answer.bullets]
    cited_facts, cited_calcs, _ = _references(answer_texts(answer))
    restated = {f for f in cited_facts + [f for c in cited_calcs for f in c.facts] if f.restated}
    caveats = list(answer.caveats) + [
        f"{f.label} for {f.period_label} was revised in a later filing: first reported as "
        f"{format_value(f.earlier_value, f.unit)}, now {format_value(f.value, f.unit)}."  # pyright: ignore[reportArgumentType]
        for f in restated
    ]
    if caveats:
        lines += ["", "**Caveats**", *(f"- {c}" for c in caveats)]
    return "\n".join(lines)

## 6. The agent

LangGraph's agent (`create_agent`) with three tools, a structured `Answer` as its final output, and
an in-memory checkpointer for follow-ups (Phase 2 swaps in the Postgres checkpointer, decision 9).
An answer that fails the citation check goes back to the model on the same thread, up to twice.
Each answer, retries included, is one MLflow trace (R-06).

In [7]:
INSTRUCTIONS = """You are the OpenSec Analyst. You answer equity analysts' questions using only \
the 10-K and 10-Q filings of the companies you can search: AMD and NVIDIA (ticker NVDA).

How to answer:
- Look facts up with get_fact and passages with search_passages. Never answer from memory.
- Write every figure as a placeholder: {fact:ID} for an XBRL fact, {calc:ID} for a \
calculation. Never type a number yourself; Python fills the figures in with unit and period. \
Python adds the period after each figure ("in FY2025", "from FY2024 to FY2025"), so don't \
write the period again next to a placeholder.
- Do all math with the compute tool, never in your head: changes, percent changes, ratios, sums.
- Every claim needs support: a {fact:ID} or {calc:ID} in its text, or citations to passages \
with a short quote copied exactly from the passage.
- Reply with a one- or two-sentence summary, then a table (for several figures or a peer \
comparison) or bullets, then caveats (such as differing fiscal calendars or definitions).
- Use each company's own fiscal labels, such as FY2025 or Q3 FY2026. NVIDIA's fiscal year ends \
in late January, so its FY2026 ended in January 2026.
- If the filings don't contain the answer, set kind to "not_disclosed". Never estimate.
- Refuse investment advice, stock prices, forecasts and anything outside the filings: set kind \
to "refusal", explain politely in the summary, and call no tools.
- For a follow-up such as "and last year?", reuse the company and measure from earlier turns.
"""


def build_tools() -> list[Any]:
    @tool
    def search_passages(
        query: str,
        ticker: str | None = None,
        form: str | None = None,
        fiscal_year: int | None = None,
    ) -> str:
        """Search filing text, filtered by ticker (AMD, NVDA), form (10-K, 10-Q), fiscal_year."""
        hits = index.search(query, ticker, form, fiscal_year)
        return "\n\n".join(f"[{p.id}] {p.chip}\n{p.text}" for p in hits) or "No passages found."

    @tool
    def get_fact(ticker: str, query: str, period: str | None = None) -> str:
        """Find XBRL facts such as revenue or net income. period is a label like FY2025, Q3 FY2026,
        9M FY2026; balance-sheet values match "end of" that period. Returns id | label | value."""
        found = facts.find(ticker, query, period)
        lines = [
            f"{f.id} | {f.label} ({f.concept}) | {format_value(f.value, f.unit)} | {f.period_label}"
            f"{' | restated' if f.restated else ''}"
            for f in found
        ]
        return "\n".join(lines) or "No matching facts."

    @tool
    def compute(operation: Operation, fact_ids: list[str]) -> str:
        """change or percent_change from the first fact to the second, ratio (first / second, as
        a percent), or sum. Returns the calculation id to cite as {calc:ID}."""
        inputs = [facts.get(i) for i in fact_ids]
        unknown = [i for i, f in zip(fact_ids, inputs, strict=True) if f is None]
        if unknown:
            return f"Unknown fact ids: {unknown}"
        try:
            calc = calculate(operation, [f for f in inputs if f])
        except ValueError as error:
            return str(error)
        calculations[calc.id] = calc
        changed = operation in ("change", "percent_change")
        shown = (
            format_change(calc.value, calc.unit) if changed else format_value(calc.value, calc.unit)
        )
        return f"{calc.id} | {shown} | {calc.period_label}"

    return [search_passages, get_fact, compute]


model = ChatOpenAI(
    model=settings.model_name,
    api_key=settings.openai_api_key,  # pyright: ignore[reportArgumentType]
    reasoning_effort=settings.model_reasoning_effort,
)
agent = create_agent(
    model,
    tools=build_tools(),
    system_prompt=INSTRUCTIONS,
    response_format=ToolStrategy(Answer),
    checkpointer=InMemorySaver(
        serde=JsonPlusSerializer(allowed_msgpack_modules=[("__main__", "Answer")])
    ),
)


@dataclass
class Result:
    question: str
    answer: Answer
    markdown: str
    problems: list[str]
    seconds: float
    tools_used: list[str]
    input_tokens: int
    output_tokens: int
    trace_id: str | None


@mlflow.trace(name="ask")
def ask(question: str, thread_id: str, retries: int = 2) -> Result:
    """One answer is one trace, retries included (R-06)."""
    config: Any = {"configurable": {"thread_id": thread_id}}
    seen = len(agent.get_state(config).values.get("messages", []))
    started = time.perf_counter()
    message = question
    for _ in range(retries + 1):
        state = agent.invoke({"messages": [HumanMessage(message)]}, config)
        answer: Answer = state["structured_response"]
        problems = check_answer(answer)
        if not problems:
            break
        message = "Your answer failed these checks. Fix them and answer again:\n- " + "\n- ".join(
            problems
        )
    seconds = time.perf_counter() - started
    turns = [m for m in state["messages"][seen:] if isinstance(m, AIMessage)]
    return Result(
        question=question,
        answer=answer,
        markdown=render_answer(answer)
        if not problems
        else "Answer rejected: " + "; ".join(problems),
        problems=problems,
        seconds=seconds,
        tools_used=[c["name"] for m in turns for c in m.tool_calls if c["name"] != "Answer"],
        input_tokens=sum(m.usage_metadata["input_tokens"] for m in turns if m.usage_metadata),
        output_tokens=sum(m.usage_metadata["output_tokens"] for m in turns if m.usage_metadata),
        trace_id=mlflow.get_active_trace_id(),
    )

## 7. Samples

The draft samples in `data/labs/samples.yaml` (plan decision 5). Expected figures are resolved
from XBRL by Python; analysts check them and set `reviewed: true`.

In [8]:
samples = yaml.safe_load((DATA_DIR / "labs" / "samples.yaml").read_text())["samples"]


def expected_facts(sample: dict[str, Any]) -> list[Fact]:
    return [facts.lookup(e["ticker"], e["concept"], e["period"]) for e in sample.get("figures", [])]


def expected_calculation(sample: dict[str, Any]) -> Calculation | None:
    spec = sample.get("calculation")
    return calculate(spec["operation"], expected_facts(sample)) if spec else None


rows = []
for sample in samples:
    for fact in expected_facts(sample):
        rows.append(
            (
                sample["id"],
                fact.ticker,
                fact.concept,
                fact.period_label,
                format_value(fact.value, fact.unit),
            )
        )
    if calc := expected_calculation(sample):
        rows.append(
            (
                sample["id"],
                "",
                calc.operation,
                calc.period_label,
                format_change(calc.value, calc.unit),
            )
        )
pd.DataFrame(rows, columns=["sample", "ticker", "concept or calculation", "period", "expected"])

,sample,ticker,concept or calculation,period,expected
0,S01,AMD,RevenueFromContractWithCustomerExcludingAssessedTax,FY2025,"$34,639.0M"
1,S02,NVDA,NetIncomeLoss,FY2026,"$120,067.0M"
2,S03,NVDA,Revenues,Q3 FY2026,"$57,006.0M"
3,S04,AMD,ResearchAndDevelopmentExpense,FY2025,"$8,091.0M"
4,S05,AMD,RevenueFromContractWithCustomerExcludingAssessedTax,FY2024,"$25,785.0M"
5,S05,AMD,RevenueFromContractWithCustomerExcludingAssessedTax,FY2025,"$34,639.0M"
6,S05,,percent_change,FY2024 to FY2025,+34.3%
7,S06,AMD,ResearchAndDevelopmentExpense,FY2025,"$8,091.0M"
8,S06,NVDA,ResearchAndDevelopmentExpense,FY2026,"$18,497.0M"
9,S07,AMD,EarningsPerShareDiluted,FY2025,$2.65


## 8. Run the samples

Each sample is an MLflow run with its trace, latency and tokens. Python scores each answer against
the brief's pass bar.

In [9]:
def cost(result: Result) -> Decimal | None:
    if settings.price_input_per_mtok is None or settings.price_output_per_mtok is None:
        return None
    return (
        result.input_tokens * settings.price_input_per_mtok
        + result.output_tokens * settings.price_output_per_mtok
    ) / 1_000_000


def cited(result: Result) -> tuple[list[Fact], list[Calculation], list[Passage]]:
    found_facts, found_calcs, _ = _references(answer_texts(result.answer))
    claims = [result.answer.summary, *result.answer.bullets]
    found_passages = [index.get(c.passage_id) for claim in claims for c in claim.citations]
    return found_facts, found_calcs, [p for p in found_passages if p]


def score(sample: dict[str, Any], result: Result) -> dict[str, bool]:
    """The brief's checks for one sample."""
    answer = result.answer
    kind = sample["kind"]
    if kind == "refusal":
        return {"refused": answer.kind == "refusal", "no tools": not result.tools_used}
    if kind == "not_disclosed":
        return {"not disclosed": answer.kind == "not_disclosed"}
    found_facts, found_calcs, found_passages = cited(result)
    checks = {
        "structured": answer.kind == "answer" and bool(answer.table or answer.bullets),
        "citations pass": not result.problems,
    }
    if sample.get("figures"):
        cited_values = {(f.ticker, f.period_label, f.value) for f in found_facts}
        cited_values |= {(f.ticker, f.period_label, f.value) for c in found_calcs for f in c.facts}
        checks["figures match XBRL"] = all(
            (f.ticker, f.period_label, f.value) in cited_values for f in expected_facts(sample)
        )
    if expected := expected_calculation(sample):
        checks["calculation"] = any(
            _rounded(c.value, "0.1") == _rounded(expected.value, "0.1") for c in found_calcs
        )
    if passage := sample.get("passage"):
        checks["cites expected Item"] = any(
            (p.ticker, p.form, p.item) == (passage["ticker"], passage["form"], passage["item"])
            for p in found_passages
        )
    return checks


def run_sample(sample: dict[str, Any], question: str, thread_id: str, name: str) -> dict[str, Any]:
    with mlflow.start_run(run_name=name):
        result = ask(question, thread_id)
        checks = score(sample, result)
        mlflow.log_params(
            {
                "question": question,
                "kind": sample["kind"],
                "model": settings.model_name,
                "reasoning": settings.model_reasoning_effort,
            }
        )
        mlflow.log_metrics(
            {
                "seconds": result.seconds,
                "input_tokens": result.input_tokens,
                "output_tokens": result.output_tokens,
                "passed": float(all(checks.values())),
            }
        )
        mlflow.log_text(result.markdown, "answer.md")
    return {
        "sample": name,
        "kind": sample["kind"],
        "passed": all(checks.values()),
        "failed checks": ", ".join(k for k, ok in checks.items() if not ok),
        "seconds": round(result.seconds, 1),
        "tools": len(result.tools_used),
        "tokens": result.input_tokens + result.output_tokens,
        "cost": cost(result),
        "trace": result.trace_id,
        "markdown": result.markdown,
    }


outcomes = []
for sample in samples:
    thread_id = str(uuid.uuid4())
    outcomes.append(run_sample(sample, sample["question"], thread_id, sample["id"]))
    if follow_up := sample.get("follow_up"):
        outcomes.append(
            run_sample(
                follow_up | {"kind": "follow_up"},
                follow_up["question"],
                thread_id,
                f"{sample['id']}-follow-up",
            )
        )

results = pd.DataFrame(outcomes)
results.drop(columns=["markdown"])

🏃 View run S01 at: http://localhost:5001/#/experiments/1/runs/a17d823144a2423f9c5d62549b4ab323
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S01-follow-up at: http://localhost:5001/#/experiments/1/runs/b93564fd83fd4af68370e7f38b34d181
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S02 at: http://localhost:5001/#/experiments/1/runs/513fcc2da1c143f88a1d7ee1d2bbe048
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S03 at: http://localhost:5001/#/experiments/1/runs/6e1fa8db27fe4f89aebc9a69c97ff89c
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S04 at: http://localhost:5001/#/experiments/1/runs/40f4cacf2470492cadc8dc2476b8b514
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S05 at: http://localhost:5001/#/experiments/1/runs/e79ce2d91ef6444f89d8535549e483c8
🧪 View experiment at: http://localhost:5001/#/experiments/1


... (5 outputs left out)

🏃 View run R01 at: http://localhost:5001/#/experiments/1/runs/9376d48304e3494ab78ac26c31b42875
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run R02 at: http://localhost:5001/#/experiments/1/runs/2c8d7d99490a423da3f1433bc4cb90b0
🧪 View experiment at: http://localhost:5001/#/experiments/1


,sample,kind,passed,failed checks,seconds,tools,tokens,cost,trace
0,S01,fact,True,,5.8,1,3130,None,tr-5f4482a7fc468a04e6e682ccbcc33992
1,S01-follow-up,follow_up,True,,3.4,1,3323,None,tr-b51edf2b4f692e2ce7cc0e1df2269f65
2,S02,fact,True,,7.7,1,5706,None,tr-126b476a0d4227f97c6e8b756fe951df
3,S03,fact,True,,5.4,1,3220,None,tr-27d3c0fc3ee791be4f29612c45ca6aa6
4,S04,fact,True,,7.7,2,10008,None,tr-df99950986bd382855f64c74226cd11b
5,S05,comparison,True,,4.9,3,3356,None,tr-268e147852a31f7a1a1a244715eeb4ed
6,S06,comparison,True,,12.4,11,26747,None,tr-fd25227975dc4c72ea152c04989ff7b6
7,S07,comparison,True,,9.9,6,18400,None,tr-6dbb022415276398ff400ff1080cf639
8,S08,risk,True,,13.9,4,12562,None,tr-ee9d277cd24a932642a7062f80e0fced
9,S09,risk,True,,17.2,5,16825,None,tr-89b33603d7a71b00067e309e440c730c


[Trace(trace_id='tr-5f4482a7fc468a04e6e682ccbcc33992'), Trace(trace_id='tr-b51edf2b4f692e2ce7cc0e1df2269f65'), Trace(trace_id='tr-126b476a0d4227f97c6e8b756fe951df'), Trace(trace_id='tr-27d3c0fc3ee791be4f29612c45ca6aa6'), Trace(trace_id='tr-df99950986bd382855f64c74226cd11b'), Trace(trace_id='tr-268e147852a31f7a1a1a244715eeb4ed'), Trace(trace_id='tr-fd25227975dc4c72ea152c04989ff7b6'), Trace(trace_id='tr-6dbb022415276398ff400ff1080cf639'), Trace(trace_id='tr-ee9d277cd24a932642a7062f80e0fced'), Trace(trace_id='tr-89b33603d7a71b00067e309e440c730c')]

## 9. The answers

Each answer as an analyst would see it, with figures filled in by Python and chips for every claim.

In [10]:
from IPython.display import Markdown, display  # noqa: E402

for outcome in outcomes:
    display(Markdown(f"### {outcome['sample']}\n\n{outcome['markdown']}"))

### S01

AMD reported revenue of $34,639.0M in FY2025. `AMD 10-K 2025 · XBRL`

- The figure is revenue from contracts with customers, excluding assessed tax: $34,639.0M in FY2025. `AMD 10-K 2025 · XBRL`

### S01-follow-up

AMD reported revenue of $25,785.0M in FY2024. `AMD 10-K 2024 · XBRL`

- This was revenue from contracts with customers, excluding assessed tax: $25,785.0M in FY2024. `AMD 10-K 2024 · XBRL`

### S02

NVIDIA reported net income of $120,067.0M in FY2026. `NVDA 10-K 2026 · XBRL`

| Measure | Amount |
| --- | --- |
| Net income | $120,067.0M (FY2026) |


### S03

NVIDIA reported revenue of $57,006.0M in Q3 FY2026. `NVDA 10-Q Q3 FY2026 · XBRL`

| Metric | Amount |
| --- | --- |
| Revenue | $57,006.0M (Q3 FY2026) |


### S04

AMD reported research and development expense of $8,091.0M in FY2025. `AMD 10-K 2025 · Item 8` `AMD 10-K 2025 · XBRL`

- AMD attributed the increase in research and development expenses primarily to higher employee-related costs from increased headcount supporting its AI strategy. `AMD 10-K 2025 · Item 7`

### S05

AMD's revenue grew +34.3% from FY2024 to FY2025. `AMD 10-K 2024 · XBRL` `AMD 10-K 2025 · XBRL`

- Revenue rose from $25,785.0M in FY2024 to $34,639.0M in FY2025. `AMD 10-K 2024 · XBRL` `AMD 10-K 2025 · XBRL`

... (3 outputs left out)

### S09

AMD’s latest annual filing says its reliance on third-party manufacturers could materially harm its business if they cannot deliver sufficient products on time using competitive technologies. `AMD 10-K 2025 · Item 1A`

- **Concentrated supply:** AMD relies on outside foundries for all product wafers and on TSMC for its most advanced microprocessor and GPU wafers; insufficient supply could lead to lost sales or strained customer relationships. `AMD 10-K 2025 · Item 1A`
- **Capacity and pricing:** Some suppliers have no long-term commitment and could prioritize other customers, increase prices, demand prepayments or reduce deliveries. `AMD 10-K 2025 · Item 1A`
- **Yields and technology:** Manufacturing or packaging problems can reduce yields or delay deliveries, while slow adoption of advanced processes could undermine AMD’s competitiveness and profitability. `AMD 10-K 2025 · Item 1A`
- **Disruptions and limited control:** AMD cites limited control over quality, schedules and inventory, as well as exposure to facility damage, power or water shortages, raw-material shortages and intellectual-property misappropriation. `AMD 10-K 2025 · Item 1A`
- **Geographic exposure:** Natural disasters and geopolitical changes affecting Taiwan could disrupt foundries and subcontractors and impede product deliveries. `AMD 10-K 2025 · Item 1A`
- **Difficulty switching and inventory commitments:** Finding alternative manufacturers can delay shipments or product introductions, while overestimating demand can leave AMD with excess inventory or higher costs under its supply commitments. `AMD 10-K 2025 · Item 1A`

**Caveats**
- These are risks described in the filing; their inclusion does not mean each has occurred.

### S10

Not disclosed in the filings.

### R01

I can’t provide NVIDIA’s current stock price because it is outside the company filings I cover.

### R02

I can’t recommend whether to buy AMD stock, but I can help analyze AMD’s financial filings.

## 10. Result

The brief's pass bar, checked in Python.

In [11]:
graded = results[results["kind"] != "refusal"]
bar = {
    "All samples pass their checks": bool(results["passed"].all()),
    "Median latency under 15 s": statistics.median(results["seconds"]) < 15,
    "Every run has an MLflow trace": bool(results["trace"].notna().all()),
}
print(
    f"Median latency: {statistics.median(results['seconds']):.1f} s; "
    f"max: {results['seconds'].max():.1f} s; tokens per answer: {int(results['tokens'].median())}"
)
pd.Series(bar, name="passed")

Median latency: 7.3 s; max: 17.2 s; tokens per answer: 5706


All samples pass their checks    True
Median latency under 15 s        True
Every run has an MLflow trace    True
Name: passed, dtype: bool